In [ ]:
!git clone https://github.com/huggingface/nanoVLM.git
%cd nanoVLM

!pip install -q datasets huggingface-hub transformers pillow tqdm thop

# Change:

- train_iter_idx += 1 - шли попорядку по массиву тренировки
- делаем проход рандомным: idx = random.randrange(len(ds_train))

- Мощнее голова: MLP + dropout

# Train code

In [ ]:
%%writefile train_aokvqa_mcq_upd.py
import argparse, math, random
from pathlib import Path
from typing import List, Dict, Any

import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

from models.vision_language_model import VisionLanguageModel
from data.processors import get_tokenizer, get_image_processor

# -----------------------------
# Специализированная MCQA-голова
# -----------------------------

class MCQHead(nn.Module):
    def __init__(self, hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        self.out = nn.Linear(hidden_dim, 4)  # 4 варианта ответа

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        # hidden: [B, T, D]
        last = hidden[:, -1, :]   # [B, D]
        last = self.dropout(last)
        return self.out(last)     # [B, 4]


# -----------------------------
# Монкипатч: forward_mcq_logits
# -----------------------------

def forward_mcq_logits(self, input_ids: torch.Tensor, images: torch.Tensor, attention_mask=None) -> torch.Tensor:
    """
    Вариант forward, который:
      - считает только скрытые состояния декодера
      - не делает проекцию на словарь
      - отдаёт логиты 4-классовой головы MCQHead
    """
    token_embd = self.decoder.token_embedding(input_ids)

    images_tensor = self._process_images(images, input_ids.device)
    if images_tensor is not None:
        image_embd = self.vision_encoder(images_tensor)
        image_embd = self.MP(image_embd)
        token_embd = self._replace_img_tokens_with_embd(input_ids, token_embd, image_embd)

    hidden, _ = self.decoder(
        token_embd,
        attention_mask=attention_mask,
        kv_cache=None,
        start_pos=0
    )  # [B, T, D]

    logits = self.mcq_head(hidden)  # [B, 4]
    return logits

# подвешиваем метод к классу
VisionLanguageModel.forward_mcq_logits = forward_mcq_logits

# -----------------------------
# Остальной тренировочный код
# -----------------------------

LETTER_TO_IDX = {"A": 0, "B": 1, "C": 2, "D": 3}
IDX_TO_LETTER = "ABCD"

def seed_everything(seed=1234):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def device_auto():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def build_mc_prompt(question: str, choices: List[str]) -> str:
    return (
        f"Question: {question}\n"
        f"A) {choices[0]}\n"
        f"B) {choices[1]}\n"
        f"C) {choices[2]}\n"
        f"D) {choices[3]}\n"
        "Answer:"
    )

class AOKVQAMCQDataset(Dataset):
    def __init__(self, split="train"):
        self.ds = load_dataset("HuggingFaceM4/A-OKVQA", split=split)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        ex = self.ds[int(idx)]
        return {
            "image": ex["image"],
            "question": ex["question"],
            "choices": list(ex["choices"]),
            "gt_idx": int(ex["correct_choice_idx"]),  # 0..3
            "qid": str(ex["question_id"])
        }

# -----------------------------
# Настройка trainable модулей
# -----------------------------

def configure_trainable_modules(model: VisionLanguageModel, args):
    """
    Настраиваем, какие части модели обучаем:
      - vision_encoder: freeze по флагу
      - MP: freeze по флагу
      - decoder: либо полностью frozen, либо размораживаем последние N слоёв
    """
    # vision
    if args.freeze_vision and hasattr(model, "vision_encoder"):
        for p in model.vision_encoder.parameters():
            p.requires_grad = False
        print("[freeze] vision_encoder")
    else:
        print("[unfreeze] vision_encoder")

    # MP
    if args.freeze_proj and hasattr(model, "MP"):
        for p in model.MP.parameters():
            p.requires_grad = False
        print("[freeze] modality projector (MP)")
    else:
        print("[unfreeze] modality projector (MP)")

    # decoder
    dec = model.decoder

    # пытаемся найти список слоёв
    if hasattr(dec, "layers"):
        layers = dec.layers
    elif hasattr(dec, "model") and hasattr(dec.model, "layers"):
        layers = dec.model.layers
    else:
        print("[warn] cannot find decoder layers (no .layers or .model.layers). Decoder untouched.")
        return

    num_layers = len(layers)
    k = max(0, int(args.unfreeze_decoder_layers))

    if k == 0:
        # все слои декодера замораживаем
        for layer in layers:
            for p in layer.parameters():
                p.requires_grad = False
        print(f"[freeze] all {num_layers} decoder layers")
    else:
        cutoff = num_layers - k
        for i, layer in enumerate(layers):
            train_this = (i >= cutoff)
            for p in layer.parameters():
                p.requires_grad = train_this
        print(f"[freeze] first {cutoff} decoder layers, [unfreeze] last {k} decoder layers")

# -----------------------------
# TRAIN
# -----------------------------

def train(args):
    seed_everything(args.seed)
    device = device_auto()
    out_dir = Path(args.output_dir); out_dir.mkdir(parents=True, exist_ok=True)

    print(f"Device: {device}")
    print(f"Loading model: {args.model_id}")
    model = VisionLanguageModel.from_pretrained(args.model_id).to(device)
    model.train()

    # сначала настраиваем, что тренируем в vision/MP/decoder
    configure_trainable_modules(model, args)

    # навешиваем MCQ-голову (всегда trainable)
    hidden_dim = model.decoder.token_embedding.embedding_dim
    model.mcq_head = MCQHead(hidden_dim, dropout=args.head_dropout).to(device)

    tokenizer = get_tokenizer(model.cfg.lm_tokenizer, model.cfg.vlm_extra_tokens, model.cfg.lm_chat_template)
    imgproc = get_image_processor(model.cfg.max_img_size, model.cfg.vit_img_size)

    ds_train = AOKVQAMCQDataset(split="train")
    ds_val   = AOKVQAMCQDataset(split="validation")

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=args.lr,
        betas=(0.9, 0.95),
        weight_decay=args.weight_decay,
    )

    total_steps = math.ceil(len(ds_train) * args.epochs / max(1, args.grad_accum))
    warmup_steps = int(args.warmup_ratio * total_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, (total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    if args.label_smoothing > 0.0:
        loss_fn = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)
        print(f"[loss] CrossEntropyLoss with label_smoothing={args.label_smoothing}")
    else:
        loss_fn = nn.CrossEntropyLoss()
        print("[loss] Plain CrossEntropyLoss (no label smoothing)")

    global_step, best_val = 0, -1.0

    def step_on_example(sample: Dict[str,Any]) -> torch.Tensor:
        img: Image.Image = sample["image"].convert("RGB")
        q, choices = sample["question"], sample["choices"]
        gt_idx = sample["gt_idx"]

        prompt = build_mc_prompt(q, choices)

        proc_img, _ = imgproc(img)
        n_imgs = proc_img.shape[0]
        num_img_tokens = model.cfg.mp_image_token_length * n_imgs
        image_tokens = tokenizer.image_token * num_img_tokens

        # только пользовательский промпт, без правильного ответа внутри контекста
        messages = [{"role": "user", "content": image_tokens + prompt}]
        ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True
        )
        input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

        logits = model.forward_mcq_logits(input_ids, proc_img.to(device))  # [1, 4]
        labels = torch.tensor([gt_idx], dtype=torch.long, device=device)   # [1]
        loss = loss_fn(logits, labels)
        return loss

    @torch.no_grad()
    def evaluate_mcq(split_ds) -> float:
        model.eval()
        correct = 0
        total = len(split_ds)
        for ex in tqdm(split_ds, total=total, desc="Eval-MCQ"):
            img: Image.Image = ex["image"].convert("RGB")
            q, choices = ex["question"], list(ex["choices"])
            gt_idx = int(ex["gt_idx"])

            proc_img, _ = imgproc(img)
            n_imgs = proc_img.shape[0]
            num_img_tokens = model.cfg.mp_image_token_length * n_imgs
            image_tokens = tokenizer.image_token * num_img_tokens

            prompt = build_mc_prompt(q, choices)
            messages = [{"role": "user", "content": image_tokens + prompt}]
            ids = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True
            )
            input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

            logits = model.forward_mcq_logits(input_ids, proc_img.to(device))  # [1, 4]
            pred_idx = int(logits.argmax(dim=-1).item())
            correct += int(pred_idx == gt_idx)

        model.train()
        return 100.0 * correct / total

    print(f"Train size: {len(ds_train)} | Val size: {len(ds_val)}")
    accum = args.grad_accum
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(range(total_steps), desc="Training", dynamic_ncols=True)
    train_iter_idx = 0

    for step in pbar:
        loss_accum = 0.0
        for _ in range(accum):
            if train_iter_idx >= len(ds_train):
                train_iter_idx = 0
            sample = ds_train[train_iter_idx]
            train_iter_idx += 1

            loss = step_on_example(sample) / accum
            loss.backward()
            loss_accum += loss.item()

        clip_grad_norm_([p for p in model.parameters() if p.requires_grad], args.grad_clip)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        global_step += 1
        pbar.set_postfix(loss=f"{loss_accum:.4f}",
                         lr=f"{scheduler.get_last_lr()[0]:.2e}")

        if (global_step % args.eval_every) == 0:
            val_acc = evaluate_mcq(ds_val)
            print(f"\n[Eval] step={global_step} | val MCQ acc = {val_acc:.2f}%")
            if val_acc > best_val:
                best_val = val_acc
                ckpt_path = out_dir / f"best_mcq_{val_acc:.2f}.pt"
                print(f"[Checkpoint] Saving to {ckpt_path}")
                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "val_acc": val_acc,
                        "step": global_step,
                    },
                    ckpt_path,
                )

    final_acc = evaluate_mcq(ds_val)
    print(f"\n[Final] val MCQ acc = {final_acc:.2f}%  (best={best_val:.2f}%)")

    full_out = out_dir / "mcq_finetuned"
    model.save_pretrained(str(full_out))
    print(f"Saved HF checkpoint to: {full_out}")

def cli():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_id", default="lusxvr/nanoVLM", help="Pretrained HF model id or local folder")
    ap.add_argument("--output_dir", default="checkpoints_mcq")
    ap.add_argument("--epochs", type=int, default=1)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--weight_decay", type=float, default=0.01)
    ap.add_argument("--warmup_ratio", type=float, default=0.10)
    ap.add_argument("--grad_accum", type=int, default=8)
    ap.add_argument("--grad_clip", type=float, default=1.0)
    ap.add_argument("--eval_every", type=int, default=500)

    ap.add_argument("--label_smoothing", type=float, default=0.0)
    ap.add_argument("--head_dropout", type=float, default=0.0)
    ap.add_argument("--unfreeze_decoder_layers", type=int, default=0)

    ap.add_argument("--freeze_vision", action="store_true", default=True)
    ap.add_argument("--freeze_proj",  action="store_true", default=False)
    ap.add_argument("--seed", type=int, default=1234)
    args = ap.parse_args()
    train(args)

if __name__ == "__main__":
    cli()


## Launch train

In [ ]:
!python train_aokvqa_mcq_upd.py \
  --model_id lusxvr/nanoVLM \
  --output_dir checkpoints_mcq_single \
  --epochs 1 \
  --lr 1e-4 \
  --grad_accum 16 \
  --label_smoothing 0.1 \
  --head_dropout 0.1 \
  --unfreeze_decoder_layers 4 \
  --freeze_vision \
  --freeze_proj


# Save

In [ ]:
import os, json

os.makedirs("/root/.config/kaggle", exist_ok=True)
creds = {
    "username": os.environ["KAGGLE_USERNAME"],
    "key": os.environ["KAGGLE_KEY"],
}
with open("/root/.config/kaggle/kaggle.json", "w") as f:
    json.dump(creds, f)
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)

print("Kaggle credentials configured from environment.")


In [ ]:
import os
import json
import subprocess
from pathlib import Path

# ===========================
# Конфигурация
# ===========================
username = os.environ["KAGGLE_USERNAME"]     # твой Kaggle username
dataset_name = "nanovlm-mcq-upd-arch-exp4-5"   # НОВОЕ имя датасета для каждой загрузки

model_dir = "checkpoints_mcq_single/mcq_finetuned"
publish_dir = f"/kaggle/working/publish_{dataset_name}"

# ===========================
# Подготовка директории
# ===========================
os.makedirs(publish_dir, exist_ok=True)

# копируем файлы модели
os.system(f"cp -r {model_dir}/* {publish_dir}/")

# ===========================
# Создаём dataset-metadata.json
# ===========================
metadata = {
    "title": f"nanoVLM MCQ Model {dataset_name}",
    "id": f"{username}/{dataset_name}",
    "licenses": [{"name": "Apache-2.0"}]
}

with open(Path(publish_dir) / "dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# ===========================
# Публикация ДАТАСЕТА (НОВОГО)
# ===========================
print("Создаём новый датасет…")

proc = subprocess.run(
    ["kaggle", "datasets", "create", "-p", publish_dir, "--dir-mode", "zip"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

print(proc.stdout)


# Inference and code

In [ ]:
import os, sys, time, json
from pathlib import Path
from typing import List

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from PIL import Image
from IPython.display import display
from tqdm.notebook import tqdm

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
    precision_score, recall_score, f1_score,
)

from thop import profile

# ======================================================
# 0. Пути и устройство
# ======================================================

MODEL_DIR = Path("/kaggle/working/nanoVLM/checkpoints_mcq_single/mcq_finetuned")  # твой чекпоинт

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# nanoVLM в PYTHONPATH
if "nanoVLM" not in sys.path:
    sys.path.append("nanoVLM")

from models.vision_language_model import VisionLanguageModel
from data.processors import get_tokenizer, get_image_processor

# ======================================================
# 1. MCQ-голова и forward_mcq_logits
# ======================================================

class MCQHead(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.out = nn.Linear(hidden_dim, 4)  # 4 варианта

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        last = hidden[:, -1, :]
        return self.out(last)

def forward_mcq_logits(self, input_ids: torch.Tensor, images: torch.Tensor, attention_mask=None) -> torch.Tensor:
    token_embd = self.decoder.token_embedding(input_ids)

    images_tensor = self._process_images(images, input_ids.device)
    if images_tensor is not None:
        image_embd = self.vision_encoder(images_tensor)
        image_embd = self.MP(image_embd)
        token_embd = self._replace_img_tokens_with_embd(input_ids, token_embd, image_embd)

    hidden, _ = self.decoder(
        token_embd,
        attention_mask=attention_mask,
        kv_cache=None,
        start_pos=0
    )
    logits = self.mcq_head(hidden)  # [B, 4]
    return logits

# навешиваем только один раз
if not getattr(VisionLanguageModel, "_mcq_patched", False):
    _orig_init = VisionLanguageModel.__init__

    def _patched_init(self, *args, **kwargs):
        _orig_init(self, *args, **kwargs)
        hidden_dim = self.decoder.token_embedding.embedding_dim
        if not hasattr(self, "mcq_head"):
            self.mcq_head = MCQHead(hidden_dim)

    VisionLanguageModel.__init__ = _patched_init
    VisionLanguageModel.forward_mcq_logits = forward_mcq_logits
    VisionLanguageModel._mcq_patched = True

# ======================================================
# 2. Загрузка модели
# ======================================================

print("Loading fine-tuned nanoVLM from:", MODEL_DIR)
model = VisionLanguageModel.from_pretrained(str(MODEL_DIR)).to(device).eval()
tokenizer = get_tokenizer(
    model.cfg.lm_tokenizer,
    model.cfg.vlm_extra_tokens,
    model.cfg.lm_chat_template,
)
imgproc = get_image_processor(model.cfg.max_img_size, model.cfg.vit_img_size)
print("Model loaded.")

# ======================================================
# 3. A-OKVQA val датасет (MCQ)
# ======================================================
from datasets import load_dataset_builder

builder = load_dataset_builder("HuggingFaceM4/A-OKVQA")
info = builder.info

download_gb = info.download_size / (1024**3)
dataset_gb  = info.dataset_size  / (1024**3)

print(f"Плановый объём скачивания (raw): {download_gb:.2f} GB")
print(f"Размер закэшированного датасета: {dataset_gb:.2f} GB")

ds_val = load_dataset("HuggingFaceM4/A-OKVQA", split="validation")
print("A-OKVQA val size:", len(ds_val))

letters = "ABCD"

def build_mc_prompt(question: str, choices: List[str]) -> str:
    return (
        f"Question: {question}\n"
        f"A) {choices[0]}\n"
        f"B) {choices[1]}\n"
        f"C) {choices[2]}\n"
        f"D) {choices[3]}\n"
        "Answer:"
    )

# ======================================================
# 4. FLOPs через thop на одном примере
# ======================================================

class MCQWrapper(nn.Module):
    def __init__(self, vlm):
        super().__init__()
        self.vlm = vlm

    def forward(self, input_ids, images):
        return self.vlm.forward_mcq_logits(input_ids, images)

mcq_wrapper = MCQWrapper(model).to(device).eval()

with torch.no_grad():
    ex0 = ds_val[0]
    img0: Image.Image = ex0["image"].convert("RGB")
    q0 = ex0["question"]
    choices0 = list(ex0["choices"])

    proc_img0, _ = imgproc(img0)
    n_imgs0 = proc_img0.shape[0]
    num_img_tokens0 = model.cfg.mp_image_token_length * n_imgs0
    image_tokens0 = tokenizer.image_token * num_img_tokens0

    prompt0 = build_mc_prompt(q0, choices0)
    messages0 = [{"role": "user", "content": image_tokens0 + prompt0}]
    ids0 = tokenizer.apply_chat_template(
        messages0,
        tokenize=True,
        add_generation_prompt=True
    )
    input_ids0 = torch.tensor(ids0, dtype=torch.long, device=device).unsqueeze(0)

    flops_per_forward, params = profile(
        mcq_wrapper,
        inputs=(input_ids0, proc_img0.to(device)),
        verbose=False
    )

print(f"FLOPs per forward_mcq_logits: {flops_per_forward:.3e}")
print(f"Params in MCQ pipeline:       {params:.3e}")

# ======================================================
# 5. Инференс на A-OKVQA + сбор метрик
# ======================================================

@torch.no_grad()
def run_nanovlm_on_aokvqa(ds, n_show_first: int = 5, max_examples: int | None = None):
    total = len(ds) if max_examples is None else min(max_examples, len(ds))

    correct = 0
    num_forwards = 0

    y_true_all = []   # 1 для GT пары, 0 для негатива
    y_score_all = []  # softmax по выбранному варианту

    latencies_e2e = []
    lat_pre = []
    lat_det = []

    pbar = tqdm(
        range(total),
        desc="Running nanoVLM on A-OKVQA (val)",
        dynamic_ncols=True,
    )

    for idx in pbar:
        t0 = time.perf_counter()

        ex = ds[idx]
        img: Image.Image = ex["image"].convert("RGB")
        q = ex["question"]
        choices = list(ex["choices"])
        gt_idx = int(ex["correct_choice_idx"])

        # препроцессинг
        t_pre0 = time.perf_counter()
        proc_img, _ = imgproc(img)
        n_imgs = proc_img.shape[0]
        num_img_tokens = model.cfg.mp_image_token_length * n_imgs
        image_tokens = tokenizer.image_token * num_img_tokens

        prompt = build_mc_prompt(q, choices)
        messages = [{"role": "user", "content": image_tokens + prompt}]
        ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True
        )
        input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
        t_pre1 = time.perf_counter()

        # forward nanoVLM
        t_det0 = time.perf_counter()
        logits = model.forward_mcq_logits(input_ids, proc_img.to(device)).squeeze(0)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        t_det1 = time.perf_counter()

        num_forwards += 1

        pred_idx = int(np.argmax(probs))
        is_correct = (pred_idx == gt_idx)
        if is_correct:
            correct += 1

        # pairwise статистика
        for j in range(len(choices)):
            y_true_all.append(1 if j == gt_idx else 0)
            y_score_all.append(float(probs[j]))

        # прогресс-бар
        seen = idx + 1
        acc_now = 100.0 * correct / seen
        pbar.set_postfix(
            correct=correct,
            wrong=seen - correct,
            acc=f"{acc_now:.2f}%",
        )

        # логирование:
        # первые n_show_first примеров — все,
        # дальше — только ошибки
        should_log = (idx < n_show_first)
        if should_log:
            print(f"\n=== Example {idx+1}/{total} ===")
            display(img)
            print("Question:", q)
            print("Choices and probs:")
            for i, (p, ch) in enumerate(zip(probs, choices)):
                letter = letters[i]
                marker_gt = "<-- GT" if i == gt_idx else ""
                marker_pred = "<-- PRED" if i == pred_idx else ""
                print(f"  {letter}) {p:.4f} | {ch} {marker_gt} {marker_pred}")
            print(f"Final predicted: {letters[pred_idx]}  (correct={is_correct})")

        # latency
        t1 = time.perf_counter()
        lat_pre.append(t_pre1 - t_pre0)
        lat_det.append(t_det1 - t_det0)
        latencies_e2e.append(t1 - t0)

    # суммарные метрики
    acc_final = 100.0 * correct / total if total > 0 else 0.0

    lat_sorted = sorted(latencies_e2e)
    p50 = lat_sorted[int(0.5 * len(lat_sorted))]
    p95 = lat_sorted[int(0.95 * len(lat_sorted))]
    total_time = sum(latencies_e2e)
    det_time = sum(lat_det)
    fps = total / total_time if total_time > 0 else 0.0

    flops_img = flops_per_forward
    total_flops = flops_per_forward * num_forwards
    tflops_effective = total_flops / det_time / 1e12 if det_time > 0 else 0.0

    print(f"\nTotal examples: {total}")
    print(f"Correct: {correct}, Wrong: {total - correct}")
    print(f"Top-1 accuracy on A-OKVQA val: {acc_final:.2f}%")

    print(f"\nE2E latency P50: {p50*1000:.2f} ms")
    print(f"E2E latency P95: {p95*1000:.2f} ms")
    print(f"E2E latency avg: {(total_time/total)*1000:.2f} ms")
    print(f"pre avg: {(sum(lat_pre)/len(lat_pre))*1000:.2f} ms")
    print(f"det avg: {(sum(lat_det)/len(lat_det))*1000:.2f} ms")
    print(f"FPS: {fps:.2f}")

    print(f"\nFLOPs per forward_mcq_logits: {flops_img:.3e}")
    print(f"Total FLOPs (all forwards):   {total_flops:.3e}")
    print(f"Effective throughput:         {tflops_effective:.3f} TFLOPs")

    return (
        np.array(y_true_all, dtype=np.int32),
        np.array(y_score_all, dtype=np.float32),
    )

# запуск оценки
y_true_nv, y_score_nv = run_nanovlm_on_aokvqa(ds_val, n_show_first=5, max_examples=None)

# ======================================================
# 6. Pairwise метрики по nanoVLM (как раньше)
# ======================================================

print("\nTotal pairs:", len(y_true_nv))
print("Positives (GT):", int(y_true_nv.sum()))
print("Negatives:", int((1 - y_true_nv).sum()))

threshold = 0.5
y_pred_nv = (y_score_nv >= threshold).astype(int)

cm_nv = confusion_matrix(y_true_nv, y_pred_nv, labels=[1, 0])

tp = cm_nv[0, 0]
fn = cm_nv[0, 1]
fp = cm_nv[1, 0]
tn = cm_nv[1, 1]

print("\nConfusion matrix (labels order: [1, 0])")
print(cm_nv)
print(f"\nTP (GT pairs predicted positive): {tp}")
print(f"FN (GT pairs predicted negative): {fn}")
print(f"FP (neg pairs predicted positive): {fp}")
print(f"TN (neg pairs predicted negative): {tn}")

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_nv,
    display_labels=["Positive (GT pair)", "Negative"],
)
fig, ax = plt.subplots(figsize=(4, 4))
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
plt.title(f"nanoVLM MCQ pairwise confusion (threshold={threshold})")
plt.show()

# ROC
fpr_nv, tpr_nv, _ = roc_curve(y_true_nv, y_score_nv)
roc_auc_nv = auc(fpr_nv, tpr_nv)

plt.figure(figsize=(4, 4))
plt.plot(fpr_nv, tpr_nv, label=f"AUC = {roc_auc_nv:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("nanoVLM MCQ ROC (pairwise, A-OKVQA)")
plt.legend()
plt.grid(True)
plt.show()

# PR
precision_nv, recall_nv, _ = precision_recall_curve(y_true_nv, y_score_nv)
ap_nv = average_precision_score(y_true_nv, y_score_nv)

plt.figure(figsize=(4, 4))
plt.plot(recall_nv, precision_nv, label=f"AP = {ap_nv:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("nanoVLM MCQ Precision–Recall (pairwise, A-OKVQA)")
plt.legend()
plt.grid(True)
plt.show()

# метрики vs threshold
thresholds = np.linspace(0.0, 1.0, 51)
accs, precs, recs, f1s = [], [], [], []

for thr in thresholds:
    y_pred_thr = (y_score_nv >= thr).astype(int)
    accs.append((y_pred_thr == y_true_nv).mean())
    precs.append(precision_score(y_true_nv, y_pred_thr, zero_division=0))
    recs.append(recall_score(y_true_nv, y_pred_thr, zero_division=0))
    f1s.append(f1_score(y_true_nv, y_pred_thr, zero_division=0))

plt.figure(figsize=(6, 4))
plt.plot(thresholds, accs, label="Accuracy")
plt.plot(thresholds, precs, label="Precision")
plt.plot(thresholds, recs, label="Recall")
plt.plot(thresholds, f1s, label="F1")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("nanoVLM MCQ (A-OKVQA): metrics vs threshold")
plt.legend()
plt.grid(True)
plt.show()

# распределение скорингов
pos_scores_nv = y_score_nv[y_true_nv == 1]
neg_scores_nv = y_score_nv[y_true_nv == 0]

plt.figure(figsize=(6, 4))
plt.hist(pos_scores_nv, bins=50, alpha=0.5, label="Positives", density=True)
plt.hist(neg_scores_nv, bins=50, alpha=0.5, label="Negatives", density=True)
plt.xlabel("Score (softmax prob of caption among 4 choices)")
plt.ylabel("Density")
plt.title("nanoVLM MCQ (A-OKVQA): score distribution")
plt.legend()
plt.show()
